# Pricing A/B Test Analysis Notebook

This notebook contains the code used to clean the pricing experiment data, answer the A/B testing practice prompts at a high level, run the statistical tests, check randomization, and export the cleaned analysis tables.

Inputs:
- `/Users/jerrywyatt/Downloads/Pricing_Test/test_results.csv`
- `/Users/jerrywyatt/Downloads/Pricing_Test/user_table.csv`
- `/Users/jerrywyatt/Downloads/A_B Testing Practice.pdf`

## 1. Setup

The original runtime did not include SciPy, so the notebook uses normal approximations for the large-sample two-sample tests and a small incomplete-gamma implementation for chi-square p-values.

In [ ]:
import json
import math
from pathlib import Path

import numpy as np
import pandas as pd
try:
    from IPython.display import display
except Exception:
    display = print

ROOT = Path("/Users/jerrywyatt/Documents/Codex/2026-07-07/files-mentioned-by-the-user-user")
OUT = ROOT / "outputs"
OUT.mkdir(exist_ok=True)

TEST_RESULTS = Path("/Users/jerrywyatt/Downloads/Pricing_Test/test_results.csv")
USER_TABLE = Path("/Users/jerrywyatt/Downloads/Pricing_Test/user_table.csv")
PDF_PATH = Path("/Users/jerrywyatt/Downloads/A_B Testing Practice.pdf")

## 2. Optional: Extract Text From The Practice PDF

This cell is optional. It extracts the PDF text so the practice guidance can be reviewed inside the notebook.

In [ ]:
try:
    import pdfplumber

    with pdfplumber.open(PDF_PATH) as pdf:
        pdf_text = "\n\n".join(page.extract_text() or "" for page in pdf.pages)
    print(pdf_text[:3000])
except Exception as exc:
    print(f"PDF extraction skipped: {exc}")

## 3. Statistical Helper Functions

In [ ]:
def norm_cdf(x):
    return 0.5 * (1 + math.erf(x / math.sqrt(2)))


def norm_p_from_z(z):
    return 2 * (1 - norm_cdf(abs(z)))


def gammaincc(a, x):
    """Regularized upper incomplete gamma Q(a, x), adapted from Numerical Recipes."""
    if x < 0 or a <= 0:
        return float("nan")
    if x == 0:
        return 1.0
    gln = math.lgamma(a)
    eps = 3e-14
    max_iter = 1000
    tiny = 1e-300

    if x < a + 1:
        ap = a
        summation = 1.0 / a
        delta = summation
        for _ in range(max_iter):
            ap += 1
            delta *= x / ap
            summation += delta
            if abs(delta) < abs(summation) * eps:
                p = summation * math.exp(-x + a * math.log(x) - gln)
                return max(0.0, min(1.0, 1.0 - p))
        p = summation * math.exp(-x + a * math.log(x) - gln)
        return max(0.0, min(1.0, 1.0 - p))

    b = x + 1 - a
    c = 1.0 / tiny
    d = 1.0 / b
    h = d
    for i in range(1, max_iter + 1):
        an = -i * (i - a)
        b += 2
        d = an * d + b
        if abs(d) < tiny:
            d = tiny
        c = b + an / c
        if abs(c) < tiny:
            c = tiny
        d = 1.0 / d
        delta = d * c
        h *= delta
        if abs(delta - 1.0) < eps:
            return max(0.0, min(1.0, math.exp(-x + a * math.log(x) - gln) * h))
    return max(0.0, min(1.0, math.exp(-x + a * math.log(x) - gln) * h))


def chi2_contingency(table):
    observed = table.to_numpy(dtype=float)
    row_sum = observed.sum(axis=1, keepdims=True)
    col_sum = observed.sum(axis=0, keepdims=True)
    total = observed.sum()
    expected = row_sum @ col_sum / total
    with np.errstate(divide="ignore", invalid="ignore"):
        chi2 = np.nansum((observed - expected) ** 2 / expected)
    dof = (observed.shape[0] - 1) * (observed.shape[1] - 1)
    p = gammaincc(dof / 2, chi2 / 2) if dof > 0 else float("nan")
    return chi2, p, dof


def two_prop_ztest(success_a, n_a, success_b, n_b):
    p_a = success_a / n_a
    p_b = success_b / n_b
    pooled = (success_a + success_b) / (n_a + n_b)
    se = math.sqrt(pooled * (1 - pooled) * (1 / n_a + 1 / n_b))
    z = (p_b - p_a) / se
    p = norm_p_from_z(z)
    se_unpooled = math.sqrt(p_a * (1 - p_a) / n_a + p_b * (1 - p_b) / n_b)
    diff = p_b - p_a
    return {
        "control_rate": p_a,
        "test_rate": p_b,
        "diff": diff,
        "relative_diff": diff / p_a,
        "z": z,
        "p_value": p,
        "ci_low": diff - 1.96 * se_unpooled,
        "ci_high": diff + 1.96 * se_unpooled,
    }


def welch_ci(a, b):
    a = np.asarray(a, dtype=float)
    b = np.asarray(b, dtype=float)
    mean_a, mean_b = a.mean(), b.mean()
    var_a, var_b = a.var(ddof=1), b.var(ddof=1)
    n_a, n_b = len(a), len(b)
    se = math.sqrt(var_a / n_a + var_b / n_b)
    df_num = (var_a / n_a + var_b / n_b) ** 2
    df_den = (var_a / n_a) ** 2 / (n_a - 1) + (var_b / n_b) ** 2 / (n_b - 1)
    df = df_num / df_den
    diff = mean_b - mean_a
    z = diff / se
    p = norm_p_from_z(z)
    return {
        "control_mean": mean_a,
        "test_mean": mean_b,
        "diff": diff,
        "relative_diff": diff / mean_a if mean_a else np.nan,
        "t": float(z),
        "p_value": float(p),
        "ci_low": diff - 1.96 * se,
        "ci_high": diff + 1.96 * se,
        "df": df,
    }


def cramers_v(table):
    chi2, p, dof = chi2_contingency(table)
    n = table.to_numpy().sum()
    r, k = table.shape
    return math.sqrt((chi2 / n) / max(1, min(k - 1, r - 1))), p, chi2, dof


def proportion_effect_size(p1, p2):
    return 2 * math.asin(math.sqrt(p1)) - 2 * math.asin(math.sqrt(p2))


def sample_size_two_proportions(p1, p2, alpha=0.05, power=0.80):
    # z values for alpha=.05 two-sided and power=.80, matching statsmodels NormalIndPower.
    z_alpha = 1.959963984540054
    z_power = 0.8416212335729143
    h = proportion_effect_size(p1, p2)
    return round(2 * ((z_alpha + z_power) / h) ** 2)

## 4. Load And Profile The Raw Data

In [ ]:
test_results = pd.read_csv(TEST_RESULTS)
users = pd.read_csv(USER_TABLE)

print("test_results shape:", test_results.shape)
print("user_table shape:", users.shape)
display(test_results.head())
display(users.head())

print("\nMissing values in test_results:")
display(test_results.isna().sum())
print("\nMissing values in user_table:")
display(users.isna().sum())

## 5. Clean And Join Data

Cleaning rules used:
- Remove rows where treatment assignment and price disagree (`test=0` should have price `39`; `test=1` should have price `59`).
- Parse timestamps and flag invalid timestamp rows. These rows are retained for the primary conversion/revenue test because assignment and outcome fields are still usable, but excluded from weekly time-trend analysis.
- Left join geography. Missing geography is retained as `Unknown` so we do not silently shrink the experiment.

In [ ]:
raw_rows = len(test_results)
raw_users = len(users)

test_results["timestamp_raw"] = test_results["timestamp"]
test_results["timestamp"] = pd.to_datetime(test_results["timestamp"], errors="coerce")
test_results["timestamp_valid"] = test_results["timestamp"].notna()
test_results["expected_price"] = np.where(test_results["test"].eq(1), 59, 39)
test_results["assignment_price_valid"] = test_results["price"].eq(test_results["expected_price"])

invalid_price = test_results.loc[~test_results["assignment_price_valid"]].copy()
invalid_timestamp = test_results.loc[~test_results["timestamp_valid"]].copy()

cleaned = test_results.loc[test_results["assignment_price_valid"]].copy()
cleaned = cleaned.merge(users, on="user_id", how="left", indicator=True)
cleaned["geo_matched"] = cleaned["_merge"].eq("both")
cleaned = cleaned.drop(columns=["_merge", "expected_price", "assignment_price_valid"])
cleaned["city"] = cleaned["city"].fillna("Unknown")
cleaned["country"] = cleaned["country"].fillna("Unknown")
cleaned["lat"] = cleaned["lat"].astype("float")
cleaned["long"] = cleaned["long"].astype("float")
cleaned["revenue"] = cleaned["price"] * cleaned["converted"]
cleaned["date"] = cleaned["timestamp"].dt.date.astype(str).replace("NaT", "")
cleaned["week"] = cleaned["timestamp"].dt.to_period("W").astype(str).replace("NaT", "")

print("Raw experiment rows:", raw_rows)
print("Invalid price rows removed:", len(invalid_price))
print("Invalid timestamp rows flagged:", len(invalid_timestamp))
print("Clean experiment rows:", len(cleaned))
print("Rows without matching geography:", int((~cleaned["geo_matched"]).sum()))

## 6. Main A/B Test

Conversion is tested as a two-proportion comparison. Revenue per visitor is tested as a Welch-style two-sample mean comparison.

In [ ]:
control = cleaned[cleaned["test"] == 0]
test = cleaned[cleaned["test"] == 1]

conversion = two_prop_ztest(
    int(control["converted"].sum()),
    len(control),
    int(test["converted"].sum()),
    len(test),
)
revenue = welch_ci(control["revenue"], test["revenue"])
conversion_t = welch_ci(control["converted"], test["converted"])

main_results = pd.DataFrame(
    [
        {
            "metric": "Conversion rate",
            "control": conversion["control_rate"],
            "test": conversion["test_rate"],
            "absolute_lift": conversion["diff"],
            "relative_lift": conversion["relative_diff"],
            "p_value": conversion["p_value"],
            "ci_low": conversion["ci_low"],
            "ci_high": conversion["ci_high"],
        },
        {
            "metric": "Revenue per visitor",
            "control": revenue["control_mean"],
            "test": revenue["test_mean"],
            "absolute_lift": revenue["diff"],
            "relative_lift": revenue["relative_diff"],
            "p_value": revenue["p_value"],
            "ci_low": revenue["ci_low"],
            "ci_high": revenue["ci_high"],
        },
    ]
)

display(main_results)

## 7. Randomization Checks

These checks compare group balance across observed covariates. With a large sample, tiny gaps can be statistically significant, so Cramer's V is used as the practical-size measure.

In [ ]:
balance_rows = []
for col in ["source", "device", "operative_system", "country", "geo_matched"]:
    table = pd.crosstab(cleaned[col], cleaned["test"])
    v, p, chi2, dof = cramers_v(table)
    balance_rows.append({"variable": col, "cramers_v": v, "p_value": p, "chi2": chi2, "dof": dof})

city_table = pd.crosstab(
    cleaned.loc[cleaned["geo_matched"], "city"],
    cleaned.loc[cleaned["geo_matched"], "test"],
)
v, p, chi2, dof = cramers_v(city_table)
balance_rows.append({"variable": "city_matched_only", "cramers_v": v, "p_value": p, "chi2": chi2, "dof": dof})

balance = pd.DataFrame(balance_rows)
display(balance)

## 8. Post-Stratified Check

This check reweights treatment and control by shared source/device/OS/geography-match strata. It is not a replacement for a perfectly randomized experiment, but it tests whether the headline direction survives basic observed-covariate adjustment.

In [ ]:
def post_stratified_effect(df, cols):
    grouped = df.groupby(cols + ["test"], dropna=False).agg(
        n=("user_id", "size"),
        conversion=("converted", "mean"),
        revenue=("revenue", "mean"),
    )
    piv = grouped.reset_index().pivot_table(index=cols, columns="test", values=["n", "conversion", "revenue"])
    piv = piv.dropna()
    weights = df.groupby(cols, dropna=False).size().reindex(piv.index) / len(df)
    weights = weights / weights.sum()
    control_conversion = float((piv[("conversion", 0)] * weights).sum())
    test_conversion = float((piv[("conversion", 1)] * weights).sum())
    control_revenue = float((piv[("revenue", 0)] * weights).sum())
    test_revenue = float((piv[("revenue", 1)] * weights).sum())
    return {
        "strata": int(len(piv)),
        "columns": cols,
        "control_conversion": control_conversion,
        "test_conversion": test_conversion,
        "conversion_lift": test_conversion - control_conversion,
        "control_revenue_per_user": control_revenue,
        "test_revenue_per_user": test_revenue,
        "revenue_lift": test_revenue - control_revenue,
    }


adjusted = post_stratified_effect(cleaned, ["source", "device", "operative_system", "geo_matched"])
adjusted

## 9. Segments And Weekly Checks

In [ ]:
def grouped_rates(df, col, max_rows=20):
    rows = []
    for value, g in df.groupby(col, dropna=False):
        control_g = g[g["test"] == 0]
        test_g = g[g["test"] == 1]
        if len(control_g) == 0 or len(test_g) == 0:
            continue
        rev = welch_ci(control_g["revenue"], test_g["revenue"])
        conv = two_prop_ztest(control_g["converted"].sum(), len(control_g), test_g["converted"].sum(), len(test_g))
        rows.append(
            {
                col: value,
                "control_n": len(control_g),
                "test_n": len(test_g),
                "control_conversion": conv["control_rate"],
                "test_conversion": conv["test_rate"],
                "conversion_lift_pp": conv["diff"],
                "control_revenue_per_user": rev["control_mean"],
                "test_revenue_per_user": rev["test_mean"],
                "revenue_lift": rev["diff"],
                "revenue_p_value": rev["p_value"],
            }
        )
    return pd.DataFrame(rows).sort_values("control_n", ascending=False).head(max_rows)


device_segments = grouped_rates(cleaned, "device")
os_segments = grouped_rates(cleaned, "operative_system")
source_segments = grouped_rates(cleaned, "source")
weekly_segments = grouped_rates(cleaned.loc[cleaned["timestamp_valid"]], "week", max_rows=20).sort_values("week")

display(device_segments)
display(source_segments.head(12))
display(weekly_segments)

## 10. Linux Sensitivity Check

The official solution flags Linux as anomalous because Linux users in the test group have 0% conversion. This cell removes Linux and reruns the revenue-per-visitor comparison.

In [ ]:
no_linux = cleaned[cleaned["operative_system"] != "linux"].copy()
no_linux_control = no_linux[no_linux["test"] == 0]
no_linux_test = no_linux[no_linux["test"] == 1]

no_linux_revenue = welch_ci(no_linux_control["revenue"], no_linux_test["revenue"])
no_linux_conversion = two_prop_ztest(
    int(no_linux_control["converted"].sum()),
    len(no_linux_control),
    int(no_linux_test["converted"].sum()),
    len(no_linux_test),
)

linux_sensitivity = pd.DataFrame(
    [
        {
            "sample": "All cleaned rows",
            "control_n": len(control),
            "test_n": len(test),
            "control_conversion": conversion["control_rate"],
            "test_conversion": conversion["test_rate"],
            "control_revenue_per_user": revenue["control_mean"],
            "test_revenue_per_user": revenue["test_mean"],
            "revenue_lift": revenue["diff"],
            "t_statistic": revenue["t"],
            "p_value": revenue["p_value"],
        },
        {
            "sample": "Linux removed",
            "control_n": len(no_linux_control),
            "test_n": len(no_linux_test),
            "control_conversion": no_linux_conversion["control_rate"],
            "test_conversion": no_linux_conversion["test_rate"],
            "control_revenue_per_user": no_linux_revenue["control_mean"],
            "test_revenue_per_user": no_linux_revenue["test_mean"],
            "revenue_lift": no_linux_revenue["diff"],
            "t_statistic": no_linux_revenue["t"],
            "p_value": no_linux_revenue["p_value"],
        },
    ]
)

print("Linux share:", (cleaned["operative_system"] == "linux").mean())
display(linux_sensitivity)

## 11. Bonus Stopping-Time Calculation

This mirrors the official solution's sample-size framing: alpha 0.05, power 0.80, and a detectable conversion drop implied by changing price from $39 to $59.

In [ ]:
raw_control_conversion = test_results.loc[test_results["test"].eq(0), "converted"].mean()
target_conversion = raw_control_conversion * 39 / 59
required_sample_per_group = sample_size_two_proportions(raw_control_conversion, target_conversion)

valid_dates = pd.to_datetime(test_results["timestamp_raw"].str.split(" ").str[0], errors="coerce")
weekly_counts = valid_dates.dt.strftime("%W").value_counts().sort_index()
middle_weekly_counts = weekly_counts.drop(weekly_counts.index[[0, weekly_counts.shape[0] - 1]])
avg_users_per_week = float(middle_weekly_counts.mean())
min_test_traffic_percent_for_two_weeks = round(required_sample_per_group / (avg_users_per_week * 2) * 100)

bonus_stopping_time = pd.DataFrame(
    [
        {
            "alpha": 0.05,
            "power": 0.80,
            "raw_control_conversion": raw_control_conversion,
            "target_conversion": target_conversion,
            "required_sample_per_group": required_sample_per_group,
            "avg_users_per_week": avg_users_per_week,
            "min_test_traffic_percent_for_two_weeks": min_test_traffic_percent_for_two_weeks,
            "recommended_min_runtime_weeks": 2,
        }
    ]
)

display(bonus_stopping_time)

## 12. Export Cleaned Data And Analysis Tables

In [ ]:
def distribution_table(col):
    table = pd.crosstab(cleaned[col], cleaned["test"], normalize="columns")
    table = table.rename(columns={0: "control_share", 1: "test_share"}).reset_index()
    table["abs_gap_pp"] = (table["test_share"] - table["control_share"]).abs()
    return table.sort_values("abs_gap_pp", ascending=False)


summary_metrics = pd.concat(
    [
        main_results,
        pd.DataFrame(
            [
                {
                    "metric": "Post-stratified conversion rate",
                    "control": adjusted["control_conversion"],
                    "test": adjusted["test_conversion"],
                    "absolute_lift": adjusted["conversion_lift"],
                    "relative_lift": adjusted["conversion_lift"] / adjusted["control_conversion"],
                    "p_value": np.nan,
                    "ci_low": np.nan,
                    "ci_high": np.nan,
                },
                {
                    "metric": "Post-stratified revenue per visitor",
                    "control": adjusted["control_revenue_per_user"],
                    "test": adjusted["test_revenue_per_user"],
                    "absolute_lift": adjusted["revenue_lift"],
                    "relative_lift": adjusted["revenue_lift"] / adjusted["control_revenue_per_user"],
                    "p_value": np.nan,
                    "ci_low": np.nan,
                    "ci_high": np.nan,
                },
            ]
        ),
    ],
    ignore_index=True,
)

summary = {
    "raw_test_rows": raw_rows,
    "raw_user_rows": raw_users,
    "removed_invalid_assignment_price_rows": len(invalid_price),
    "flagged_invalid_timestamp_rows": len(invalid_timestamp),
    "clean_rows": len(cleaned),
    "unmatched_geo_rows_clean": int((~cleaned["geo_matched"]).sum()),
    "control_n": len(control),
    "test_n": len(test),
    "control_conversions": int(control["converted"].sum()),
    "test_conversions": int(test["converted"].sum()),
    "conversion": conversion,
    "conversion_t_test": {"t": conversion_t["t"], "p_value": conversion_t["p_value"]},
    "revenue": revenue,
    "post_stratified_source_device_os_geo": adjusted,
    "linux_sensitivity": {
        "linux_share": float((cleaned["operative_system"] == "linux").mean()),
        "rows_after_removing_linux": int(len(no_linux)),
        "control_n": int(len(no_linux_control)),
        "test_n": int(len(no_linux_test)),
        "control_conversion": no_linux_conversion["control_rate"],
        "test_conversion": no_linux_conversion["test_rate"],
        "control_revenue_per_user": no_linux_revenue["control_mean"],
        "test_revenue_per_user": no_linux_revenue["test_mean"],
        "revenue_lift": no_linux_revenue["diff"],
        "t_statistic": no_linux_revenue["t"],
        "p_value": no_linux_revenue["p_value"],
    },
    "bonus_stopping_time": {
        "alpha": 0.05,
        "power": 0.80,
        "raw_control_conversion": float(raw_control_conversion),
        "target_conversion": float(target_conversion),
        "required_sample_per_group": int(required_sample_per_group),
        "avg_users_per_week": avg_users_per_week,
        "min_test_traffic_percent_for_two_weeks": min_test_traffic_percent_for_two_weeks,
        "recommended_min_runtime_weeks": 2,
    },
    "date_min": cleaned["timestamp"].min().strftime("%Y-%m-%d"),
    "date_max": cleaned["timestamp"].max().strftime("%Y-%m-%d"),
}

cleaned.to_csv(OUT / "cleaned_pricing_ab_data.csv", index=False)
invalid_price.to_csv(OUT / "removed_invalid_price_rows.csv", index=False)
invalid_timestamp.to_csv(OUT / "flagged_invalid_timestamp_rows.csv", index=False)
summary_metrics.to_csv(OUT / "summary_metrics.csv", index=False)
balance.to_csv(OUT / "balance_tests.csv", index=False)
distribution_table("source").to_csv(OUT / "source_distribution_gaps.csv", index=False)
distribution_table("device").to_csv(OUT / "device_distribution_gaps.csv", index=False)
distribution_table("operative_system").to_csv(OUT / "os_distribution_gaps.csv", index=False)
distribution_table("geo_matched").to_csv(OUT / "geo_match_distribution.csv", index=False)
distribution_table("city").head(30).to_csv(OUT / "top_city_distribution_gaps.csv", index=False)
device_segments.to_csv(OUT / "device_segments.csv", index=False)
os_segments.to_csv(OUT / "os_segments.csv", index=False)
source_segments.to_csv(OUT / "source_segments.csv", index=False)
weekly_segments.to_csv(OUT / "weekly_segments.csv", index=False)
linux_sensitivity.to_csv(OUT / "linux_sensitivity.csv", index=False)
bonus_stopping_time.to_csv(OUT / "bonus_stopping_time.csv", index=False)

with open(OUT / "analysis_summary.json", "w") as f:
    json.dump(summary, f, indent=2)

print(f"Saved outputs to {OUT}")

## 13. Conclusion

The higher `$59` price reduces conversion but increases revenue per visitor. The Linux sensitivity check strengthens the revenue result, and the bonus sample-size section suggests the test had enough volume. Because the experiment still has some observed randomization imbalance, especially across city among matched users, the report recommendation is to treat the result as promising and validate it with stratified randomization or a gradual rollout with holdout monitoring.